In [2]:
rm(list = ls())
options(stringsAsFactors = FALSE)

In [3]:
library(MetaNeighbor)
library(SingleCellExperiment)
library(magrittr)
library(Matrix.utils)
library(dplyr)
library(tidyr)
library(stringr)
library(tibble)
library(reshape2)

Loading required package: SummarizedExperiment
Loading required package: MatrixGenerics
Loading required package: matrixStats

Attaching package: 'MatrixGenerics'

The following objects are masked from 'package:matrixStats':

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOrderStats, rowProds, rowQuantiles, rowRanges, ro

In [4]:
source("./functions/2017-08-28-runMN-US.R")
source("./functions/2017-08-28-runMN-US.pearson.R")

In [5]:
step <- function(msg) {
    cat(sprintf("\n[%s] %s\n", format(Sys.time(), "%H:%M:%S"), msg))
    flush.console()
}

In [ ]:
# ---- parameters ------------------------------------------------------------
#! Acer & Amel
# dataset.A <- "Acer"
# col.A  <- "leiden_res0.50"
# dataset.B <- "Amel"
# col.B  <- "cell_type_cluster"

# A_pb_file <- "./matrix/MN_R_dataset/Acer_raw_8910.RDS"
# B_pb_file <- "./matrix/MN_R_dataset/Amel_raw_8910.RDS"
# A_marker_file <- "./metadata/MN_R_markers/Acer_markers_8910.tsv"
# B_marker_file <- "./metadata/MN_R_markers/Amel_markers_8910.tsv"
# ortho_file <- "./orthologs/Acer_vs_Amel_121_4mn.tsv"

# B_sp_name <- "Apis_mellifera"

# oup_name <- paste0(dataset.A, "-", col.A, "_vs_", dataset.B, "-", col.B)
# out_dir  <- file.path("./metadata/MN_R_auroc", oup_name)
# dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

#! Acer & Hsal
# dataset.A <- "Acer"
# col.A  <- "leiden_res0.50"
# dataset.B <- "Hsal"
# col.B  <- "cluster_name"

# A_pb_file <- "./matrix/MN_R_dataset/Acer_raw_8073.RDS"
# B_pb_file <- "./matrix/MN_R_dataset/Hsal_raw_8073.RDS"
# A_marker_file <- "./metadata/MN_R_markers/Acer_markers_8073.tsv"
# B_marker_file <- "./metadata/MN_R_markers/Hsal_markers_8073.tsv"
# ortho_file <- "./orthologs/Acer_vs_Hsal_121_4mn.tsv"

# B_sp_name <- "Harpegnathos_saltator"

# oup_name <- paste0(dataset.A, "-", col.A, "_vs_", dataset.B, "-", col.B)
# out_dir  <- file.path("./metadata/MN_R_auroc", oup_name)
# dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

#! Acer & Dmel
dataset.A <- "Acer"
col.A  <- "leiden_res0.50"
dataset.B <- "Dmel"
col.B  <- "afca_annotation"

A_pb_file <- "./matrix/MN_R_dataset/Acer_raw_6171.RDS"
B_pb_file <- "./matrix/MN_R_dataset/Dmel_raw_6171_onlybrain.RDS"
A_marker_file <- "./metadata/MN_R_markers/Acer_markers_6171.tsv"
B_marker_file <- "./metadata/MN_R_markers/Dmel_markers_6171_onlybrain.tsv"
ortho_file <- "./orthologs/Acer_vs_Dmel_121_4mn(AFCA).tsv"

B_sp_name <- "Drosophila_melanogaster"

oup_name <- paste0(dataset.A, "-", col.A, "_vs_", dataset.B, "-", col.B)
out_dir  <- file.path("./metadata/MN_R_auroc", oup_name)
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

In [8]:
# ---- 1. read pseudo-cell SCE and marker tables -----------------------------
step("1. read pseudo-cell SCE and markers")
sce.A <- readRDS(A_pb_file)
sce.B <- readRDS(B_pb_file)
gene.A <- read.delim(A_marker_file)
gene.B <- read.delim(B_marker_file)

cat("A(Acer) pseudo-cells:", ncol(sce.A), " genes:", nrow(sce.A), "\n")
cat("B pseudo-cells:", ncol(sce.B), " genes:", nrow(sce.B), "\n")
cat("A(Acer) marker rows :", nrow(gene.A), "\n")
cat("B marker rows :", nrow(gene.B), "\n")


[15:28:06] 1. read pseudo-cell SCE and markers
A(Acer) pseudo-cells: 5835  genes: 8073 
B pseudo-cells: 1868  genes: 8073 
A(Acer) marker rows : 7274 
B marker rows : 4444 


In [ ]:
# ---- 2. one-to-one ortholog alignment (Bsp -> Acer gene IDs) -------------
step("2. align one-to-one orthologs (Bsp -> Acer)")
ortho <- read.delim(ortho_file)
stopifnot(all(c("Apis_cerana", B_sp_name) %in% colnames(ortho)))

B_ids <- ortho[[B_sp_name]] #! 注：这里 B_sp_name 是字符串 "Harpegnathos_saltator"，不能像 ortho$Apis_cerana 一样使用 ortho$B_sp_name，但 ortho[[B_sp_name]] 可以
stopifnot(!is.null(B_ids))

stopifnot(!anyDuplicated(ortho$Apis_cerana),
            !anyDuplicated(B_ids))
b2a <- setNames(ortho$Apis_cerana, B_ids)

cat("   common genes before alignment:",
    length(intersect(rownames(sce.A), rownames(sce.B))), "\n")

idx <- match(rownames(sce.B), names(b2a))
stopifnot(!anyNA(idx))
rownames(sce.B) <- unname(b2a[rownames(sce.B)])

common_genes <- intersect(rownames(sce.A), rownames(sce.B))
cat("   common genes after alignment :", length(common_genes), "\n")
stopifnot(length(common_genes) == nrow(sce.A),
            length(common_genes) == nrow(sce.B))
sce.B <- sce.B[rownames(sce.A), ]

idx.b <- match(gene.B$gene, names(b2a))
cat("   Bsp marker genes without mapping:", sum(is.na(idx.b)), "\n")
gene.B$gene <- unname(b2a[gene.B$gene])
gene.B <- gene.B[!is.na(gene.B$gene), ]
gene.A <- gene.A[!is.na(gene.A$gene), ]
gene.A <- gene.A[gene.A$gene %in% common_genes, ]
gene.B <- gene.B[gene.B$gene %in% common_genes, ]


[15:36:14] 2. align one-to-one orthologs (Bsp -> Acer)


   common genes before alignment: 0 
   common genes after alignment : 8073 
   Bsp marker genes without mapping: 0 


In [28]:
# ---- 3. prefix cell names and merge SCE -----------------------------------
step("3. merge Acer and Bsp pseudo-cell SCE")
colnames(sce.A) <- paste0(dataset.A, ".", colnames(sce.A))
colnames(sce.B) <- paste0(dataset.B, ".", colnames(sce.B))
data <- list(A = sce.A, B = sce.B)
data.merge <- mergeSCE(data)
stopifnot(identical(colnames(data.merge), c(colnames(sce.A), colnames(sce.B))))
cat("   merged SCE:", paste(dim(data.merge), collapse = " x "), "\n")

data.merge$study_id <- str_split_fixed(colnames(data.merge), "\\.", 2)[, 1]
data.merge$cell_type <- paste0(data.merge$study_id, ".", data.merge$cell_type)
cat("   study_id:", paste(unique(data.merge$study_id), collapse = ", "), "\n")
cat("   cell types:", length(unique(data.merge$cell_type)), "\n")


[15:40:56] 3. merge Acer and Bsp pseudo-cell SCE


   merged SCE: 8073 x 7703 
   study_id: Acer, Hsal 
   cell types: 66 


In [31]:
# ---- 4. merged CP10K + z-score (same as reference 02) ---------------------
step("4. CP10K + z-score on merged data")
data.mat <- counts(data.merge)
data.scale <- apply(X = data.mat, MARGIN = 2,
                    FUN = function(x) { x * 1000 / sum(x) })
data.zscore <- apply(X = data.scale, MARGIN = 1,
                    FUN = function(x) {
                        if (sd(x) != 0) (x - mean(x)) / sd(x) else x - mean(x)
                    })
data.zscore <- t(data.zscore)
cat("   z-score dim:", paste(dim(data.zscore), collapse = " x "),
    "(genes x pseudo-cells)\n")


[15:41:33] 4. CP10K + z-score on merged data
   z-score dim: 8073 x 7703 (genes x pseudo-cells)


In [32]:
# ---- 5. phenotype table ---------------------------------------------------
step("5. build phenotype table")
pheno <- colData(data.merge) %>%
    as.data.frame() %>%
    rownames_to_column(var = "Sample_ID") %>%
    select(Sample_ID, study_id, cell_type) %>%
    rename(Study_ID = study_id, Celltype = cell_type)
celltypes <- unique(pheno$Celltype)
cat("pheno rows:", nrow(pheno), "\n")
cat("unique cell types:", length(celltypes), "\n")


[15:43:37] 5. build phenotype table


pheno rows: 7703 
unique cell types: 66 


In [34]:
# ---- 6. marker gene selection (reference 02) ------------------------------
step("6. select top 30 markers per cluster")
A.gene <- gene.A %>%
    filter(p_val_adj < 0.05, avg_log2FC >= 1.25) %>%
    group_by(cluster) %>%
    top_n(n = 30, wt = avg_log2FC) %>%
    as.data.frame() %>%
    select(gene)

B.gene <- gene.B %>%
    filter(p_val_adj < 0.05, avg_log2FC >= 1.25) %>%
    group_by(cluster) %>%
    top_n(n = 30, wt = avg_log2FC) %>%
    as.data.frame() %>%
    select(gene)

marker.genes <- union(A.gene$gene, B.gene$gene)
marker.genes <- intersect(marker.genes, rownames(data.zscore))
cat("   Acer selected markers:", length(unique(A.gene$gene)), "\n")
cat("   Bsp selected markers:", length(unique(B.gene$gene)), "\n")
cat("   union markers used     :", length(marker.genes), "\n")


[15:44:10] 6. select top 30 markers per cluster


   Acer selected markers: 913 
   Bsp selected markers: 701 
   union markers used     : 1475 


In [39]:
# ---- 7. MetaNeighbor AUROC (reference 02) ---------------------------------
step("7. run MetaNeighbor Spearman")
AUROC.matrix.s <- run_MetaNeighbor_s(marker.genes, data.zscore, celltypes, pheno)

step("8. run MetaNeighbor Pearson")
AUROC.matrix.p <- run_MetaNeighbor_p(marker.genes, data.zscore, celltypes, pheno)


[15:45:57] 7. run MetaNeighbor Spearman



[15:49:27] 8. run MetaNeighbor Pearson


In [40]:
# ---- 8. reshape and write auroc.tsv ---------------------------------------
step("9. reshape and write auroc.tsv")
AUROC.data.s <- reshape2::melt(AUROC.matrix.s, value.name = "AUROCs")
AUROC.data.p <- reshape2::melt(AUROC.matrix.p, value.name = "AUROCp")
res <- merge(AUROC.data.p, AUROC.data.s, by = c("Var1", "Var2"))
out_file <- file.path(out_dir, "auroc.tsv")
write.table(res, out_file, sep = "\t", quote = FALSE, row.names = FALSE)

cat("result dim:", paste(dim(res), collapse = " x "), "\n")
cat("saved:", out_file, "\n")
step("done")


[15:53:27] 9. reshape and write auroc.tsv


result dim: 4356 x 4 
saved: ./metadata/MN_R_auroc/Acer-leiden_res0.50_vs_Hsal-cluster_name/auroc.tsv 

[15:53:28] done
